# Structured credit 2 — CLO equity (residual) tranche

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/instruments/structured_credit_clo_debt.ipynb`.

The **equity** (or *residual*) tranche is the **first-loss**, most-junior piece of
the CLO. It receives **excess spread** — collateral interest left after the debt
coupons and fees — plus whatever par the manager builds. That makes it a **levered,
long-credit, long-reinvestment** position whose value swings far more with default,
prepayment and the **reinvestment price** than any debt tranche.


## Concept

- **Residual cashflow:** equity is paid *last* in the waterfall; its value is the PV
  of excess spread plus terminal par.
- **Reinvestment price is an equity story:** buying loans below par (**97**) builds
  par the equity ultimately keeps — so the residual is highly sensitive to it.
- **Convex to defaults:** a little default just trims excess spread; enough default
  eats into equity principal. **Break-even CDR ≈ 0** because equity is first-loss.
- We sweep **default and prepayment assumptions and curves**, and stress on a
  **CPR × CDR × severity** grid.

In [ ]:
import json
import pandas as pd
from datetime import date

from finstack_quant.core.market_data import DiscountCurve, ForwardCurve, MarketContext
from finstack_quant.valuations.instruments.fixed_income import StructuredCredit

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
AS_OF = "2024-06-15"
AS_OF_D = date(2024, 6, 15)
print("Imports OK (structured credit).")

In [ ]:
# --- thin builders over the structured-credit serde spec (Rust is canonical) ---
def money(x):
    return {"amount": str(x), "currency": "USD"}

def asset(aid, atype, bal, rate, spread_bps=None, index_id=None,
          maturity="2031-06-15", day_count="Act360"):
    """One pool asset. Floating loans set spread_bps + index_id (rate = fallback)."""
    return {"id": aid, "asset_type": atype, "balance": money(bal), "rate": rate,
            "spread_bps": spread_bps, "index_id": index_id, "maturity": maturity,
            "credit_quality": None, "industry": None, "obligor_id": None,
            "is_defaulted": False, "recovery_amount": None, "purchase_price": None,
            "acquisition_date": None, "day_count": day_count,
            "smm_override": None, "mdr_override": None}

def floating(spread_bp, index_id="USD-SOFR-3M"):
    """Minimal floating tranche coupon (index + spread); other fields take engine defaults."""
    return {"Floating": {"index_id": index_id, "spread_bp": spread_bp,
            "reset_freq": {"count": 3, "unit": "months"}, "dc": "Act360", "calendar_id": "nyse"}}

def fixed(rate):
    return {"Fixed": {"rate": rate}}

def tranche(tid, att, det, seniority, bal, coupon, priority, freq_n=3, maturity="2031-06-15"):
    """One liability tranche. attachment/detachment are 0-100 (percent of structure)."""
    return {"id": tid, "attachment_point": att, "detachment_point": det,
            "behavior_type": "Standard", "seniority": seniority, "rating": None,
            "original_balance": money(bal), "current_balance": money(bal), "target_balance": None,
            "coupon": coupon, "oc_trigger": None, "ic_trigger": None,
            "credit_enhancement": {"subordination": money(0), "overcollateralization": money(0),
                                   "reserve_account": money(0), "excess_spread": 0.0,
                                   "cash_trap_active": False},
            "frequency": {"count": freq_n, "unit": "months"}, "day_count": "Act360",
            "deferred_interest": money(0), "pik_enabled": False, "is_revolving": False,
            "can_reinvest": False, "maturity": maturity, "expected_maturity": None,
            "payment_priority": priority, "attributes": {}}

def pool(pid, deal_type, assets, reinvest_end=None):
    """Collateral pool. reinvest_end (a date string) opens a reinvestment period."""
    rp = None
    if reinvest_end is not None:
        rp = {"end_date": reinvest_end, "is_active": True,
              "criteria": {"max_price": 101.0, "min_yield": 0.0,
                           "maintain_credit_quality": True, "maintain_wal": True,
                           "apply_eligibility_criteria": True}}
    return {"id": pid, "deal_type": deal_type, "base_currency": "USD", "assets": assets,
            "cumulative_defaults": money(0), "cumulative_recoveries": money(0),
            "cumulative_prepayments": money(0), "cumulative_scheduled_amortization": None,
            "reinvestment_period": rp, "collection_account": money(0),
            "reserve_account": money(0), "excess_spread_account": money(0), "rep_lines": None}

def build(did, deal_type, pool_dict, tranches, freq_n, closing, first_pay, maturity,
          reinvest_end=None, prepay=None, default=None, recovery=None, reinvestment_price=None):
    """Assemble + return (StructuredCredit, spec). reinvestment_price is % of par (e.g. 97)."""
    prepay = prepay if prepay is not None else {"cpr": 0.0, "curve": None}
    default = default if default is not None else {"cdr": 0.0, "curve": None}
    recovery = recovery if recovery is not None else {"rate": 0.40, "recovery_lag": 0}
    total = sum(int(t["original_balance"]["amount"]) for t in tranches)
    spec = {"id": did, "deal_type": deal_type, "pool": pool_dict,
            "tranches": {"tranches": tranches, "total_size": money(total)},
            "closing_date": closing, "first_payment_date": first_pay,
            "reinvestment_end_date": reinvest_end, "maturity": maturity,
            "frequency": {"count": freq_n, "unit": "months"},
            "payment_calendar_id": "nyse", "discount_curve_id": "USD-OIS",
            "pricing_overrides": {"mc_paths": 1}, "attributes": {},
            "prepayment_spec": prepay, "default_spec": default, "recovery_spec": recovery,
            "stochastic_prepay_spec": {"model": "Deterministic", **prepay},
            "stochastic_default_spec": {"model": "Deterministic", **default},
            "market_conditions": {"refi_rate": 0.04, "original_rate": None, "hpa": None,
                                  "unemployment": None, "seasonal_factor": 1.0, "custom_factors": {}},
            "credit_factors": {"credit_score": None, "dti": None, "ltv": None,
                               "delinquency_days": 0, "unemployment_rate": None, "custom_factors": {}}}
    if reinvestment_price is not None:
        spec["behavior_overrides"] = {"reinvestment_price": reinvestment_price}
    d = StructuredCredit(spec=spec)
    d.validate()
    return d, spec

## Market context

In [ ]:
# OIS discount curve (PV) + 3M-SOFR forward curve (projects floating coupons).
ois = DiscountCurve("USD-OIS", AS_OF_D,
                    [(0.0, 1.0), (1.0, 0.95), (3.0, 0.86), (5.0, 0.78), (8.0, 0.66), (10.0, 0.60)],
                    day_count="act_365f")
sofr = ForwardCurve("USD-SOFR-3M", 0.25,
                    [(0.0, 0.053), (3.0, 0.045), (8.0, 0.040)],
                    base_date=AS_OF_D, day_count="act_360")
market_json = MarketContext().insert(ois).insert(sofr).to_json()
print("Market:", [c.get("id") for c in json.loads(market_json)["curves"]])

In [ ]:
def tranche_results(deal):
    """Per-tranche Monte-Carlo results (npv, average_life, credit_duration, expected_loss)."""
    out = json.loads(deal.price(market_json, AS_OF, "structured_credit_stochastic"))
    return out["details"]["data"]["tranche_results"]

def price_table(deal, spec):
    """Tidy DataFrame of PV, price (% of par), WAL and expected loss per tranche."""
    orig = {t["id"]: int(t["original_balance"]["amount"]) for t in spec["tranches"]["tranches"]}
    rows = []
    for t in tranche_results(deal):
        ob = orig[t["tranche_id"]]
        pv = float(t["npv"]["amount"])
        rows.append({"tranche": t["tranche_id"], "seniority": t["seniority"],
                     "orig_$mm": ob / 1e6, "PV_$mm": pv / 1e6,
                     "price_%par": 100 * pv / ob if ob else float("nan"),
                     "WAL_yrs": t["average_life"], "credit_dur": t["credit_duration"],
                     "exp_loss_$mm": float(t["expected_loss"]["amount"]) / 1e6})
    return pd.DataFrame(rows)

In [ ]:
def build_clo(reinvest_end="2027-06-15", reinvestment_price=None,
              cdr=0.02, cpr=0.0, severity=0.40, recovery_lag=3, prepay_curve=None):
    assets = [
        asset("LOAN_A", {"type": "FirstLienLoan", "industry": "Technology"},
              250_000_000, 0.0575, spread_bps=450.0, index_id="USD-SOFR-3M"),
        asset("LOAN_B", {"type": "FirstLienLoan", "industry": "Healthcare"},
              250_000_000, 0.0525, spread_bps=425.0, index_id="USD-SOFR-3M"),
    ]
    tranches = [
        tranche("CLASS-A", 0.0, 80.0, "Senior", 400_000_000, floating(150.0), 1),
        tranche("CLASS-B", 80.0, 92.0, "Mezzanine", 60_000_000, floating(600.0), 2),
        tranche("EQUITY", 92.0, 100.0, "Equity", 40_000_000, fixed(0.0), 3),
    ]
    prepay = {"cpr": cpr, "curve": prepay_curve}
    return build("CLO-EQ", "CLO", pool("POOL-CLO", "CLO", assets, reinvest_end),
                 tranches, 3, "2024-06-15", "2024-09-15", "2031-06-15",
                 reinvest_end=reinvest_end, reinvestment_price=reinvestment_price,
                 prepay=prepay, default={"cdr": cdr, "curve": None},
                 recovery={"rate": 1.0 - severity, "recovery_lag": recovery_lag})

EQUITY_NOTIONAL = 40_000_000

def equity_npv(**kw):
    d, s = build_clo(**kw)
    eq = next(t for t in tranche_results(d) if t["tranche_id"] == "EQUITY")
    return float(eq["npv"]["amount"])

## The equity tranche at base assumptions

In [ ]:
clo, clo_spec = build_clo()
pt = price_table(clo, clo_spec)
eq_npv = pt.set_index("tranche").loc["EQUITY", "PV_$mm"] * 1e6
print(f"Equity NPV: ${eq_npv:,.0f}  ({eq_npv / EQUITY_NOTIONAL:.2f}x its ${EQUITY_NOTIONAL/1e6:.0f}mm notional), "
      f"WAL {pt.set_index('tranche').loc['EQUITY','WAL_yrs']:.2f}y")
pt

## Reinvestment: off / par / 97 / 100

The headline equity sensitivity. Reinvesting principal at a **97** price builds
extra par each period, lifting the residual materially above the par (100) case;
turning reinvestment off shortens the deal and gives up that upside.

In [ ]:
rows = []
for label, reinv, px in [("no reinvestment", None, None),
                         ("reinvest @ par (100)", "2027-06-15", 100.0),
                         ("reinvest @ 99", "2027-06-15", 99.0),
                         ("reinvest @ 97", "2027-06-15", 97.0)]:
    npv = equity_npv(reinvest_end=reinv, reinvestment_price=px, cpr=0.20)
    rows.append({"scenario": label, "equity_NPV_$mm": npv / 1e6,
                 "x_notional": npv / EQUITY_NOTIONAL})
pd.DataFrame(rows)

## Default & prepayment assumption sweeps

Equity is **short credit** (NPV falls as CDR rises) and **short prepayment** (faster
CPR returns principal early and shrinks the reinvestment/excess-spread window).

In [ ]:
cdr_grid = [0.00, 0.01, 0.02, 0.04, 0.06, 0.08]
cpr_grid = [0.00, 0.10, 0.20, 0.30]
by_cdr = pd.DataFrame({"CDR": cdr_grid,
                       "equity_NPV_$mm": [equity_npv(cdr=c, cpr=0.20) / 1e6 for c in cdr_grid]})
by_cpr = pd.DataFrame({"CPR": cpr_grid,
                       "equity_NPV_$mm": [equity_npv(cdr=0.02, cpr=c) / 1e6 for c in cpr_grid]})
print("Equity NPV vs constant CDR (CPR=20%):"); display(by_cdr)
print("Equity NPV vs constant CPR (CDR=2%):"); display(by_cpr)

### A prepayment *curve* (PSA ramp)

Beyond constant speeds, the engine supports named ramps. Here a **150% PSA** curve
(prepayment seasoning ramp) is applied to the same equity.

In [ ]:
npv_psa = equity_npv(cdr=0.02, prepay_curve={"curve": "psa", "speed_multiplier": 1.5})
npv_const = equity_npv(cdr=0.02, cpr=0.06)
pd.DataFrame([{"prepay": "constant 6% CPR", "equity_NPV_$mm": npv_const / 1e6},
              {"prepay": "150% PSA ramp", "equity_NPV_$mm": npv_psa / 1e6}])

## Break-even CDR (first-loss check)

In [ ]:
print("Equity break-even CDR:", f"{clo.breakeven_cdr(market_json, AS_OF, 'EQUITY') * 100:.3f}%",
      "(≈0 — equity is first-loss)")
print("AAA   break-even CDR:", f"{clo.breakeven_cdr(market_json, AS_OF, 'CLASS-A') * 100:.2f}%")

## Equity scenario grid (CPR × CDR × severity)

The residual's full stress surface: price (% of its own notional) collapses as
default and severity climb.

In [ ]:
grid = json.dumps({"cprs": [0.10, 0.20], "cdrs": [0.00, 0.02, 0.04, 0.06],
                   "severities": [0.40, 0.60], "recovery_lag": 3})
cells = json.loads(clo.scenario_table(market_json, AS_OF, "EQUITY", grid))["cells"]
sc = pd.DataFrame(cells)
sc.pivot_table(index="cdr", columns=["cpr", "severity"], values="price").rename_axis(
    index="CDR").style.format("{:.0f}").set_caption("Equity price (% of equity notional)")

## Takeaways

- The CLO **equity** is a **levered residual**: small in notional, large in NPV
  swing. Its **break-even CDR ≈ 0** confirms first-loss status.
- It is **short credit and short prepayment** — NPV falls with CDR and with faster CPR.
- The **reinvestment price** is the equity's signature lever: reinvesting at **97**
  builds par that lifts the residual well above the par-reinvestment case.